In [19]:
import pandas as pd 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment
import plotly.express as px

In [20]:
Ja = pd.read_csv("celtics_data/players_advanced.csv")
Jn = pd.read_csv("celtics_data/players_traditional.csv")
Ea = pd.read_csv("celtics_data/team_advanced.csv")
En = pd.read_csv("celtics_data/team_traditional.csv")

In [21]:
STATS = ["PTS", "REB", "AST", "STL", "BLK", "FG_PCT", "FG3_PCT"]
PERCENT_STATS = {"FG_PCT", "FG3_PCT"}

In [22]:
# Nombres  para los títulos de cada subplot
LABELS = {
    "PTS": "PTS per game",
    "REB": "Reb per game",
    "AST": "Ast per game",
    "STL": "Stl per game",
    "BLK": "Blk per game",
    "FG_PCT": "FG%",
    "FG3_PCT": "3FG%",
}

top_todos = {
    col: Jn.nlargest(5, col)[["PLAYER_NAME", col]]
    for col in STATS
}

N_COLS = 2
N_ROWS = -(-len(STATS) // N_COLS) 

fig = make_subplots(
    rows=N_ROWS,
    cols=N_COLS,
    subplot_titles=[LABELS[s] for s in STATS],
    horizontal_spacing=0.15,
    vertical_spacing=0.10,
)

for i, stat in enumerate(STATS):
    row = i // N_COLS + 1
    col = i % N_COLS + 1

    df_stat = top_todos[stat].sort_values(stat, ascending=True) 

    if stat in PERCENT_STATS:
        text_labels = [f"{v:.1%}" for v in df_stat[stat]]
    else:
        text_labels = [f"{v:.1f}" for v in df_stat[stat]]

    fig.add_trace(
        go.Bar(
            x=df_stat[stat],
            y=df_stat["PLAYER_NAME"],
            orientation="h",
            text=text_labels,
            textposition="outside",
            marker_color="#007A33",   # verde Celtics
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    max_val = df_stat[stat].max()
    fig.update_xaxes(range=[0, max_val * 1.25], row=row, col=col, showticklabels=False)
    fig.update_yaxes(row=row, col=col, automargin=True)

# Estilo general (limpio, pensado para imprimir)
fig.update_layout(
    title_text="Boston Celtics Top 5 in traditional Stats",
    title_x=0.5,
    template="plotly_white",
    height=1100,
    width=950,
    font=dict(family="Arial", size=13, color="black"),
    margin=dict(t=100, l=20, r=20, b=20),
)

In [23]:
Advanced_Stats = ["OFF_RATING","DEF_RATING","NET_RATING","TS_PCT"]

In [24]:
LABELS = {
    "OFF_RATING": "Offensive Rating",
    "DEF_RATING": "Defensive Rating",
    "NET_RATING": "Net Rating",
    "TS_PCT": "True Shooting",
}

HIGHER_IS_BETTER = {
    "OFF_RATING": True,
    "DEF_RATING": False,   
    "NET_RATING": True,
    "TS_PCT": True,
}

PERCENT_STATS = {"TS_PCT"}
top_todos_advanced = {}
for col in Advanced_Stats:
    if HIGHER_IS_BETTER[col]:
        top_todos_advanced[col] = Ja.nlargest(5, col)[["PLAYER_NAME", col]]
    else:
        top_todos_advanced[col] = Ja.nsmallest(5, col)[["PLAYER_NAME", col]]

N_COLS = 2
N_ROWS = -(-len(Advanced_Stats) // N_COLS)

fig = make_subplots(
    rows=N_ROWS,
    cols=N_COLS,
    subplot_titles=[LABELS[s] for s in Advanced_Stats],
    horizontal_spacing=0.15,
    vertical_spacing=0.10,
)

for i, stat in enumerate(Advanced_Stats):
    row = i // N_COLS + 1
    col = i % N_COLS + 1

    ascending = HIGHER_IS_BETTER[stat]
    df_stat = top_todos_advanced[stat].sort_values(stat, ascending=ascending)

    if stat in PERCENT_STATS:
        text_labels = [f"{v:.1%}" for v in df_stat[stat]]
    else:
        text_labels = [f"{v:.1f}" for v in df_stat[stat]]

    fig.add_trace(
        go.Bar(
            x=df_stat[stat],
            y=df_stat["PLAYER_NAME"],
            orientation="h",
            text=text_labels,
            textposition="outside",
            marker_color="#007A33",
            showlegend=False,
        ),
        row=row,
        col=col,
    )


    if HIGHER_IS_BETTER[stat]:
        max_val = df_stat[stat].max()
        fig.update_xaxes(range=[0, max_val * 1.25], row=row, col=col, showticklabels=False)
    else:
        max_val = df_stat[stat].max()
        fig.update_xaxes(range=[0, max_val * 1.15], row=row, col=col, showticklabels=False)

    fig.update_yaxes(row=row, col=col, automargin=True)


fig.update_layout(
    title_text="Boston Celtics — Top 5 en Estadísticas Avanzadas (2024-25)",
    title_x=0.5,
    template="plotly_white",
    height=650,
    width=950,
    font=dict(family="Arial", size=13, color="black"),
    margin=dict(t=100, l=20, r=20, b=20),
)


fig.add_annotation(
    text="* En Defensive Rating, menor valor = mejor defensa",
    xref="paper", yref="paper",
    x=0.5, y=-0.08,
    showarrow=False,
    font=dict(size=11, color="gray"),
)

In [25]:
MIN_GP = 16
MIN_MPG = 12 

Ja_filtrado = Ja[(Ja["GP"] >= MIN_GP) & (Ja["MIN"] >= MIN_MPG)].copy()

print(f"Jugadores considerados: {len(Ja_filtrado)} de {len(Ja)} "
      f"(filtro: GP >= {MIN_GP}, MIN >= {MIN_MPG})")


def make_top10_table(df, stat, title, ascending, filename=None, value_fmt="{:.1f}"):
    cols = ["PLAYER_NAME", "GP", "MIN", stat]
    df_top = df.sort_values(stat, ascending=ascending).head(10)[cols].reset_index(drop=True)
    df_top.insert(0, "Rank", range(1, len(df_top) + 1))

    values_formatted = [str(v) for v in df_top["Rank"]]
    gp_formatted = df_top["GP"].astype(int).astype(str).tolist()
    min_formatted = [f"{v:.1f}" for v in df_top["MIN"]]
    stat_formatted = [value_fmt.format(v) for v in df_top[stat]]

    n = len(df_top)
    row_colors = ["#D9F2E6" if i < 5 else "white" for i in range(n)]

    fig = go.Figure(
        data=[
            go.Table(
                columnwidth=[50, 220, 60, 80, 110],
                header=dict(
                    values=["#", "Jugador", "PJ", "MIN", title],
                    fill_color="#007A33",
                    font=dict(color="white", size=13, family="Arial"),
                    align="center",
                    height=32,
                ),
                cells=dict(
                    values=[
                        df_top["Rank"],
                        df_top["PLAYER_NAME"],
                        gp_formatted,
                        min_formatted,
                        stat_formatted,
                    ],
                    fill_color=[row_colors] * 5,
                    font=dict(color="black", size=12, family="Arial"),
                    align=["center", "left", "center", "center", "center"],
                    height=28,
                ),
            )
        ]
    )

    fig.update_layout(
        title_text=f"Boston Celtics — Top 10 {title} (2024-25)",
        title_x=0.5,
        template="plotly_white",
        width=650,
        height=430,
        margin=dict(t=60, l=10, r=10, b=10),
    )

    fig.show()
    return df_top


top10_def = make_top10_table(
    Ja_filtrado, "DEF_RATING", "Defensive Rating",
    ascending=True,   # menor es mejor
    filename=None,
)

top10_off = make_top10_table(
    Ja_filtrado, "OFF_RATING", "Offensive Rating",
    ascending=False,  # mayor es mejor
    filename=None,
)

top10_net = make_top10_table(
    Ja_filtrado, "NET_RATING", "Net Rating",
    ascending=False,  # mayor es mejor
    filename=None,
)


MIN_FG3A = 1.5  # intentos de triple por partido mínimos, para que el % sea representativo
Jn_filtrado = Jn[Jn["FG3A"] >= MIN_FG3A].copy()

print(f"\nJugadores considerados para FG3_PCT: {len(Jn_filtrado)} de {len(Jn)} "
      f"(filtro: FG3A >= {MIN_FG3A} por partido)")

top10_fg3 = make_top10_table(
    Jn_filtrado, "FG3_PCT", "3PT %",
    ascending=False,  # mayor es mejor
    filename=None,
    value_fmt="{:.1%}",
)

Jugadores considerados: 11 de 17 (filtro: GP >= 16, MIN >= 12)



Jugadores considerados para FG3_PCT: 13 de 17 (filtro: FG3A >= 1.5 por partido)


In [26]:
df = Ja.merge(
    Jn[["PLAYER_ID", "FGA"]],
    on="PLAYER_ID",
    how="left",
)
df = df.rename(columns={"FGA": "FGA_PG"})


# Filtrar jugadores de rotación
MIN_GP = 16
MIN_MPG = 12

df = df[(df["GP"] >= MIN_GP) & (df["MIN"] >= MIN_MPG)].reset_index(drop=True)
print(f"Jugadores de rotación incluidos en el modelo: {len(df)}")


# Definir variables por rol y preparar la matriz de features
ROLE_VARS = {
    "Defensive Anchor": ["DREB_PCT", "REB_PCT", "DEF_RATING"],
    "Floor Spacer / Efficiency": ["EFG_PCT", "TS_PCT", "OFF_RATING"],
    "Playmaker": ["AST_PCT", "AST_TO", "AST_RATIO"],
    "Primary Scorer": ["USG_PCT", "FGA_PG", "PIE"],
}

FEATURE_COLS = list(dict.fromkeys(
    col for cols in ROLE_VARS.values() for col in cols
))

X_raw = df[FEATURE_COLS].copy()


# Estandarizar (y voltear el signo de DEF_RATING)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled = pd.DataFrame(X_scaled, columns=FEATURE_COLS, index=df.index)
X_scaled["DEF_RATING"] = -X_scaled["DEF_RATING"]

# PCA (2 componentes, solo para visualización)
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df["PC1"] = pca_coords[:, 0]
df["PC2"] = pca_coords[:, 1]

var_exp = pca.explained_variance_ratio_
print(f"Varianza explicada: PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%}, "
      f"total={var_exp.sum():.1%}")

# KMeans (k=4) sobre la matriz estandarizada completa
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# 7. Score compuesto por rol (media de las 3 variables estandarizadas
#    de cada rol) — se usa tanto para asignar cluster -> rol como para
#    ordenar el Top 3 dentro de cada arquetipo.
for role, cols in ROLE_VARS.items():
    df[f"score_{role}"] = X_scaled[cols].mean(axis=1)

# Matriz clusters x roles con el score promedio de cada combinación
cluster_ids = sorted(df["Cluster"].unique())
score_matrix = np.array([
    [df.loc[df["Cluster"] == c, f"score_{role}"].mean() for role in ROLE_VARS]
    for c in cluster_ids
])

# Asignación óptima 1-a-1 cluster -> rol (maximiza el score total,
# evita que dos roles distintos queden mapeados al mismo cluster)
row_ind, col_ind = linear_sum_assignment(-score_matrix)
roles_list = list(ROLE_VARS.keys())
cluster_to_role = {cluster_ids[r]: roles_list[c] for r, c in zip(row_ind, col_ind)}

df["Rol"] = df["Cluster"].map(cluster_to_role)

print("\nMapeo Cluster -> Rol asignado:")
for c, role in cluster_to_role.items():
    n_players = (df["Cluster"] == c).sum()
    print(f"  Cluster {c} -> {role} ({n_players} jugadores)")

# Top 3 de cada arquetipo (dentro de su cluster asignado, ordenado
#por el score compuesto de las variables que definen ese rol)
print("TOP 3 POR ARQUETIPO")

top3_por_rol = {}
for role in ROLE_VARS:
    subset = df[df["Rol"] == role].copy()
    subset = subset.sort_values(f"score_{role}", ascending=False).head(3)
    top3_por_rol[role] = subset

    print(f"{role}")
    cols_show = ["PLAYER_NAME"] + ROLE_VARS[role] + [f"score_{role}"]
    print(subset[cols_show].to_string(index=False))

# Visualización PCA
fig = px.scatter(
    df,
    x="PC1",
    y="PC2",
    color="Rol",
    text="PLAYER_NAME",
    title="Boston Celtics — Perfiles Funcionales (PCA + KMeans, 2025-26)",
    template="plotly_white",
    width=900,
    height=650,
)
fig.update_traces(textposition="top center", marker=dict(size=10))
fig.show()

Jugadores de rotación incluidos en el modelo: 11
Varianza explicada: PC1=42.8%, PC2=23.3%, total=66.1%

Mapeo Cluster -> Rol asignado:
  Cluster 0 -> Primary Scorer (3 jugadores)
  Cluster 1 -> Defensive Anchor (4 jugadores)
  Cluster 2 -> Playmaker (2 jugadores)
  Cluster 3 -> Floor Spacer / Efficiency (2 jugadores)
TOP 3 POR ARQUETIPO
Defensive Anchor
      PLAYER_NAME  DREB_PCT  REB_PCT  DEF_RATING  score_Defensive Anchor
    Hugo González     0.158    0.107       106.2                0.548610
Baylor Scheierman     0.138    0.089       109.0               -0.082551
     Jordan Walsh     0.146    0.108       112.8               -0.271104
Floor Spacer / Efficiency
  PLAYER_NAME  EFG_PCT  TS_PCT  OFF_RATING  score_Floor Spacer / Efficiency
   Luka Garza    0.654   0.682       119.4                         1.293126
Neemias Queta    0.654   0.674       118.8                         1.139078
Playmaker
     PLAYER_NAME  AST_PCT  AST_TO  AST_RATIO  score_Playmaker
Payton Pritchard    0.234 

In [ ]:
"""
Full analysis of the Boston Celtics (2024-25 season).

Figures/tables generated (each in its own variable, so you can call
them separately whenever you want):
    fig_traditional  -> Top 5 in traditional stats (subplots)
    fig_advanced     -> Top 5 in advanced stats (subplots)
    table_def        -> Top 10 Defensive Rating
    table_off        -> Top 10 Offensive Rating
    table_net        -> Top 10 Net Rating
    table_fg3        -> Top 10 3PT%
    fig_pca          -> PCA + KMeans scatter of functional player profiles (shown at the end)

Requirements:
    pip install pandas plotly kaleido scikit-learn scipy
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment

# ========================================================================
# 0. Load data
# ========================================================================
Ja = pd.read_csv("celtics_data/players_advanced.csv")
Jn = pd.read_csv("celtics_data/players_traditional.csv")
Ea = pd.read_csv("celtics_data/team_advanced.csv")
En = pd.read_csv("celtics_data/team_traditional.csv")


# ========================================================================
# 1. FIG_TRADITIONAL — Top 5 in traditional stats
# ========================================================================
STATS_TRAD = ["PTS", "REB", "AST", "STL", "BLK", "FG_PCT", "FG3_PCT"]
PERCENT_STATS_TRAD = {"FG_PCT", "FG3_PCT"}
LABELS_TRAD = {
    "PTS": "PTS per game",
    "REB": "Reb per game",
    "AST": "Ast per game",
    "STL": "Stl per game",
    "BLK": "Blk per game",
    "FG_PCT": "FG%",
    "FG3_PCT": "3FG%",
}

top_todos = {
    col: Jn.nlargest(5, col)[["PLAYER_NAME", col]]
    for col in STATS_TRAD
}

N_COLS = 2
N_ROWS = -(-len(STATS_TRAD) // N_COLS)

fig_traditional = make_subplots(
    rows=N_ROWS,
    cols=N_COLS,
    subplot_titles=[LABELS_TRAD[s] for s in STATS_TRAD],
    horizontal_spacing=0.15,
    vertical_spacing=0.10,
)

for i, stat in enumerate(STATS_TRAD):
    row = i // N_COLS + 1
    col = i % N_COLS + 1

    df_stat = top_todos[stat].sort_values(stat, ascending=True)

    if stat in PERCENT_STATS_TRAD:
        text_labels = [f"{v:.1%}" for v in df_stat[stat]]
    else:
        text_labels = [f"{v:.1f}" for v in df_stat[stat]]

    fig_traditional.add_trace(
        go.Bar(
            x=df_stat[stat],
            y=df_stat["PLAYER_NAME"],
            orientation="h",
            text=text_labels,
            textposition="outside",
            marker_color="#007A33",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    max_val = df_stat[stat].max()
    fig_traditional.update_xaxes(range=[0, max_val * 1.25], row=row, col=col, showticklabels=False)
    fig_traditional.update_yaxes(row=row, col=col, automargin=True)

fig_traditional.update_layout(
    title_text="Boston Celtics Top 5 in traditional Stats",
    title_x=0.5,
    template="plotly_white",
    height=1100,
    width=950,
    font=dict(family="Arial", size=13, color="black"),
    margin=dict(t=100, l=20, r=20, b=20),
)


# ========================================================================
# 2. FIG_ADVANCED — Top 5 in advanced stats
# ========================================================================
Advanced_Stats = ["OFF_RATING", "DEF_RATING", "NET_RATING", "TS_PCT"]
LABELS_ADV = {
    "OFF_RATING": "Offensive Rating",
    "DEF_RATING": "Defensive Rating",
    "NET_RATING": "Net Rating",
    "TS_PCT": "True Shooting",
}
HIGHER_IS_BETTER = {
    "OFF_RATING": True,
    "DEF_RATING": False,
    "NET_RATING": True,
    "TS_PCT": True,
}
PERCENT_STATS_ADV = {"TS_PCT"}

top_todos_advanced = {}
for col in Advanced_Stats:
    if HIGHER_IS_BETTER[col]:
        top_todos_advanced[col] = Ja.nlargest(5, col)[["PLAYER_NAME", col]]
    else:
        top_todos_advanced[col] = Ja.nsmallest(5, col)[["PLAYER_NAME", col]]

N_COLS = 2
N_ROWS = -(-len(Advanced_Stats) // N_COLS)

fig_advanced = make_subplots(
    rows=N_ROWS,
    cols=N_COLS,
    subplot_titles=[LABELS_ADV[s] for s in Advanced_Stats],
    horizontal_spacing=0.15,
    vertical_spacing=0.10,
)

for i, stat in enumerate(Advanced_Stats):
    row = i // N_COLS + 1
    col = i % N_COLS + 1

    ascending = HIGHER_IS_BETTER[stat]
    df_stat = top_todos_advanced[stat].sort_values(stat, ascending=ascending)

    if stat in PERCENT_STATS_ADV:
        text_labels = [f"{v:.1%}" for v in df_stat[stat]]
    else:
        text_labels = [f"{v:.1f}" for v in df_stat[stat]]

    fig_advanced.add_trace(
        go.Bar(
            x=df_stat[stat],
            y=df_stat["PLAYER_NAME"],
            orientation="h",
            text=text_labels,
            textposition="outside",
            marker_color="#007A33",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    max_val = df_stat[stat].max()
    if HIGHER_IS_BETTER[stat]:
        fig_advanced.update_xaxes(range=[0, max_val * 1.25], row=row, col=col, showticklabels=False)
    else:
        fig_advanced.update_xaxes(range=[0, max_val * 1.15], row=row, col=col, showticklabels=False)

    fig_advanced.update_yaxes(row=row, col=col, automargin=True)

fig_advanced.update_layout(
    title_text="Boston Celtics — Top 5 in Advanced Stats (2024-25)",
    title_x=0.5,
    template="plotly_white",
    height=650,
    width=950,
    font=dict(family="Arial", size=13, color="black"),
    margin=dict(t=100, l=20, r=20, b=20),
)
fig_advanced.add_annotation(
    text="* For Defensive Rating, lower value = better defense",
    xref="paper", yref="paper",
    x=0.5, y=-0.08,
    showarrow=False,
    font=dict(size=11, color="gray"),
)


# ========================================================================
# 3. TABLES — Top 10 Def/Off/Net Rating and Top 10 3PT%
#    (built here but NOT called/shown)
# ========================================================================
MIN_GP = 16
MIN_MPG = 12

Ja_filtered = Ja[(Ja["GP"] >= MIN_GP) & (Ja["MIN"] >= MIN_MPG)].copy()
print(f"Players considered (rating tables): {len(Ja_filtered)} of {len(Ja)} "
      f"(filter: GP >= {MIN_GP}, MIN >= {MIN_MPG})")


def make_top10_table(df, stat, title, ascending, value_fmt="{:.1f}"):
    """Builds the table and returns it as a Plotly figure (without showing it)."""
    cols = ["PLAYER_NAME", "GP", "MIN", stat]
    df_top = df.sort_values(stat, ascending=ascending).head(10)[cols].reset_index(drop=True)
    df_top.insert(0, "Rank", range(1, len(df_top) + 1))

    gp_formatted = df_top["GP"].astype(int).astype(str).tolist()
    min_formatted = [f"{v:.1f}" for v in df_top["MIN"]]
    stat_formatted = [value_fmt.format(v) for v in df_top[stat]]

    n = len(df_top)
    row_colors = ["#D9F2E6" if i < 5 else "white" for i in range(n)]

    fig_table = go.Figure(
        data=[
            go.Table(
                columnwidth=[50, 220, 60, 80, 110],
                header=dict(
                    values=["#", "Player", "GP", "MIN", title],
                    fill_color="#007A33",
                    font=dict(color="white", size=13, family="Arial"),
                    align="center",
                    height=32,
                ),
                cells=dict(
                    values=[
                        df_top["Rank"],
                        df_top["PLAYER_NAME"],
                        gp_formatted,
                        min_formatted,
                        stat_formatted,
                    ],
                    fill_color=[row_colors] * 5,
                    font=dict(color="black", size=12, family="Arial"),
                    align=["center", "left", "center", "center", "center"],
                    height=28,
                ),
            )
        ]
    )

    fig_table.update_layout(
        title_text=f"Boston Celtics — Top 10 {title} (2024-25)",
        title_x=0.5,
        template="plotly_white",
        width=650,
        height=430,
        margin=dict(t=60, l=10, r=10, b=10),
    )
    return fig_table


table_def = make_top10_table(Ja_filtered, "DEF_RATING", "Defensive Rating", ascending=True)
table_off = make_top10_table(Ja_filtered, "OFF_RATING", "Offensive Rating", ascending=False)
table_net = make_top10_table(Ja_filtered, "NET_RATING", "Net Rating", ascending=False)

MIN_FG3A = 1.5
Jn_filtered = Jn[Jn["FG3A"] >= MIN_FG3A].copy()
print(f"Players considered (FG3_PCT table): {len(Jn_filtered)} of {len(Jn)} "
      f"(filter: FG3A >= {MIN_FG3A} per game)")

table_fg3 = make_top10_table(
    Jn_filtered, "FG3_PCT", "3PT %", ascending=False, value_fmt="{:.1%}"
)

# Note: table_def, table_off, table_net and table_fg3 are ready but
# NOT shown here. When you want to see them: table_def.show(), etc.


# ========================================================================
# 4. PCA + KMeans — Functional player profiles and Top 3 by archetype
# ========================================================================
df = Ja.merge(
    Jn[["PLAYER_ID", "FGA"]],
    on="PLAYER_ID",
    how="left",
)
df = df.rename(columns={"FGA": "FGA_PG"})

df = df[(df["GP"] >= MIN_GP) & (df["MIN"] >= MIN_MPG)].reset_index(drop=True)
print(f"Rotation players included in the PCA/KMeans model: {len(df)}")

ROLE_VARS = {
    "Defensive Anchor": ["DREB_PCT", "REB_PCT", "DEF_RATING"],
    "Floor Spacer / Efficiency": ["EFG_PCT", "TS_PCT", "OFF_RATING"],
    "Playmaker": ["AST_PCT", "AST_TO", "AST_RATIO"],
    "Primary Scorer": ["USG_PCT", "FGA_PG", "PIE"],
}

FEATURE_COLS = list(dict.fromkeys(
    col for cols in ROLE_VARS.values() for col in cols
))

X_raw = df[FEATURE_COLS].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled = pd.DataFrame(X_scaled, columns=FEATURE_COLS, index=df.index)
X_scaled["DEF_RATING"] = -X_scaled["DEF_RATING"]

pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df["PC1"] = pca_coords[:, 0]
df["PC2"] = pca_coords[:, 1]

var_exp = pca.explained_variance_ratio_
print(f"Explained variance: PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%}, "
      f"total={var_exp.sum():.1%}")

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

for role, cols in ROLE_VARS.items():
    df[f"score_{role}"] = X_scaled[cols].mean(axis=1)

cluster_ids = sorted(df["Cluster"].unique())
score_matrix = np.array([
    [df.loc[df["Cluster"] == c, f"score_{role}"].mean() for role in ROLE_VARS]
    for c in cluster_ids
])

row_ind, col_ind = linear_sum_assignment(-score_matrix)
roles_list = list(ROLE_VARS.keys())
cluster_to_role = {cluster_ids[r]: roles_list[c] for r, c in zip(row_ind, col_ind)}
df["Role"] = df["Cluster"].map(cluster_to_role)

print("\nCluster -> Role mapping:")
for c, role in cluster_to_role.items():
    n_players = (df["Cluster"] == c).sum()
    print(f"  Cluster {c} -> {role} ({n_players} players)")

print("\nTOP 3 BY ARCHETYPE")
top3_por_rol = {}
for role in ROLE_VARS:
    subset = df[df["Role"] == role].copy()
    subset = subset.sort_values(f"score_{role}", ascending=False).head(3)
    top3_por_rol[role] = subset

    print(f"\n{role}")
    cols_show = ["PLAYER_NAME"] + ROLE_VARS[role] + [f"score_{role}"]
    print(subset[cols_show].to_string(index=False))

fig_pca = px.scatter(
    df,
    x="PC1",
    y="PC2",
    color="Role",
    text="PLAYER_NAME",
    title="Boston Celtics — Functional Player Profiles (PCA + KMeans, 2024-25)",
    template="plotly_white",
    width=900,
    height=650,
)
fig_pca.update_traces(textposition="top center", marker=dict(size=10))

# Only this figure is auto-shown when the script runs
fig_pca.show()


# ========================================================================
# 5. FIG_USAGE_EFFICIENCY — Usage% vs True Shooting% quadrant matrix
# ========================================================================
# Reuses Ja_filtered (rotation players) from section 3.
usg_mean = Ja_filtered["USG_PCT"].mean()
ts_mean = Ja_filtered["TS_PCT"].mean()

fig_usage_efficiency = px.scatter(
    Ja_filtered,
    x="USG_PCT",
    y="TS_PCT",
    text="PLAYER_NAME",
    size="MIN",
    size_max=22,
    color_discrete_sequence=["#007A33"],
    title="Boston Celtics — Usage Rate vs. True Shooting (2024-25)",
    labels={"USG_PCT": "Usage % (USG%)", "TS_PCT": "True Shooting % (TS%)"},
    template="plotly_white",
    width=900,
    height=700,
)
fig_usage_efficiency.update_traces(
    textposition="top center",
    marker=dict(line=dict(width=1, color="white")),
)

Players considered (rating tables): 11 of 17 (filter: GP >= 16, MIN >= 12)
Players considered (FG3_PCT table): 13 of 17 (filter: FG3A >= 1.5 per game)
Rotation players included in the PCA/KMeans model: 11
Explained variance: PC1=42.8%, PC2=23.3%, total=66.1%

Cluster -> Role mapping:
  Cluster 0 -> Primary Scorer (3 players)
  Cluster 1 -> Defensive Anchor (4 players)
  Cluster 2 -> Playmaker (2 players)
  Cluster 3 -> Floor Spacer / Efficiency (2 players)

TOP 3 BY ARCHETYPE

Defensive Anchor
      PLAYER_NAME  DREB_PCT  REB_PCT  DEF_RATING  score_Defensive Anchor
    Hugo González     0.158    0.107       106.2                0.548610
Baylor Scheierman     0.138    0.089       109.0               -0.082551
     Jordan Walsh     0.146    0.108       112.8               -0.271104

Floor Spacer / Efficiency
  PLAYER_NAME  EFG_PCT  TS_PCT  OFF_RATING  score_Floor Spacer / Efficiency
   Luka Garza    0.654   0.682       119.4                         1.293126
Neemias Queta    0.654   0.674

In [28]:
fig_usage_efficiency.show()